<a href="https://colab.research.google.com/github/abishekr19/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Unit of analysis: One row represents one webpage for one month.
Time window: I will use historical monthly webpage data to predict whether a webpage is likely to show a declining search-performance trend in a future period.

In [10]:
import os
import getpass
import duckdb

def get_hf_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

token = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected successfully!")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected successfully!
Feature window: February 2026
Label window: March 2026


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: Search performance metrics such as impressions, clicks, CTR, and average position.

Label: Whether the webpage shows a declining search-performance trend in the future.

Context: Webpage URL/page identifier and month.

Excluded: Future-period performance values, because they would not be available when making the prediction and would cause data leakage.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# 1. Verify the grain: one row = one client × content × daily record
grain_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS unique_client_content
FROM {FEB}
""").df()

display(grain_check)


# 2. Verify February row count and shape
shape_check = con.execute(f"""
SELECT
    COUNT(*) AS february_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM {FEB}
""").df()

display(shape_check)


# 3. Verify GSC availability using IS TRUE
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows
FROM {FEB}
""").df()

display(availability_check)

,total_rows,unique_client_content
0,7355108,321546


,february_rows,clients,content_items
0,7355108,54,321546


,total_rows,available_rows
0,7355108,2621783


In [14]:
# Build the February analysis universe and five-feature frame

feature_df = con.execute(f"""
WITH feb_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb,

        SUM(gsc_avg_position * gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        )
        / NULLIF(
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            ), 0
        ) AS avg_position_feb,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_days_available_feb

    FROM {FEB}
    GROUP BY
        client_hash_id,
        content_hash_id
),

universe AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.impressions_feb,
        f.clicks_feb,
        f.avg_position_feb,
        f.gsc_days_available_feb
    FROM feb_agg f
    INNER JOIN {DIM} c
        ON f.content_hash_id = c.content_hash_id
    WHERE f.impressions_feb >= 100
      AND f.clicks_feb >= 3
      AND c.is_published IS TRUE
      AND c.content_created_date <= DATE '2026-02-28'
)

SELECT
    client_hash_id,
    content_hash_id,
    impressions_feb,
    clicks_feb,
    clicks_feb / NULLIF(impressions_feb, 0) AS ctr_feb,
    avg_position_feb,
    gsc_days_available_feb
FROM universe
""").df()

display(feature_df.head(10))

print("Feature dataframe shape:", feature_df.shape)
print("Feature rows:", len(feature_df))

,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,gsc_days_available_feb
0,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,0.008186,6.316508,28
1,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,0.001024,41.814739,28
2,client_e547b89c05043229,content_babd931911c9ee33,2680.0,31.0,0.011567,5.049254,28
3,client_e547b89c05043229,content_431784c057b25a5d,3641.0,6.0,0.001648,8.829992,28
4,client_e547b89c05043229,content_7275e583711cf60b,2371.0,6.0,0.002531,7.358077,28
5,client_e547b89c05043229,content_2caf3716bd6e42bc,798.0,3.0,0.003759,10.511278,28
6,client_e547b89c05043229,content_c104e7e26ad25b73,1351.0,3.0,0.002221,8.779423,28
7,client_e547b89c05043229,content_e24453f672279e47,3940.0,6.0,0.001523,34.369797,28
8,client_e547b89c05043229,content_e87512f3582b175c,676.0,3.0,0.004438,13.934911,28
9,client_e547b89c05043229,content_810323d88df953e8,4917.0,19.0,0.003864,4.794387,28


Feature dataframe shape: (29700, 7)
Feature rows: 29700


In [15]:
# Create the March 2026 label

label_df = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS imp_mar,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clk_mar,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS measured_days_mar

FROM {MAR}

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

# Combine February features with March outcome
frame = feature_df.merge(
    label_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

frame["measured_days_mar"] = (
    frame["measured_days_mar"]
    .fillna(0)
    .astype(int)
)

# Keep only pages with measured March data
frame = frame[
    frame["measured_days_mar"] > 0
].copy()

frame[["imp_mar", "clk_mar"]] = frame[
    ["imp_mar", "clk_mar"]
].fillna(0)

# Create the March label
frame["went_dark"] = (
    frame["clk_mar"] == 0
).astype(int)

print("Final frame shape:", frame.shape)
print("Positive labels:", frame["went_dark"].sum())
print("Positive rate:", frame["went_dark"].mean())

Final frame shape: (29353, 11)
Positive labels: 1159
Positive rate: 0.03948489081184206


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Five honest February features
honest_features = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "gsc_days_available_feb"
]

X = frame[honest_features].fillna(0)
y = frame["went_dark"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

honest_auc = roc_auc_score(
    y_test,
    model.predict_proba(X_test)[:, 1]
)

print("Honest ROC-AUC:", honest_auc)

Honest ROC-AUC: 0.8222124928456469


In [17]:
# DELIBERATE LEAKAGE EXPERIMENT
# clk_mar comes from the future March label window.
# It should NOT be available at the February decision moment.

leaky_features = honest_features + ["clk_mar"]

X_leaky = frame[leaky_features].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_auc = roc_auc_score(
    y_test,
    leaky_model.predict_proba(X_test)[:, 1]
)

print("Honest ROC-AUC:", honest_auc)
print("Leaky ROC-AUC:", leaky_auc)

Honest ROC-AUC: 0.8222124928456469
Leaky ROC-AUC: 1.0


### Leakage experiment

The honest model uses only February features and achieved a ROC-AUC of 0.8184.

For the leakage experiment, `clk_mar` was intentionally added as a feature. This value comes from the March label window and therefore would not be available at the February decision moment.

The leaky model produces a much higher, near-perfect score because it has access to future information that directly relates to the label. This demonstrates why label-derived or future-derived columns must not be used as features.

The honest ROC-AUC of 0.8184 is the valid number to keep.

In [18]:
# Remove the deliberately leaked future feature

final_features = honest_features.copy()

print("Final features:")
for feature in final_features:
    print("-", feature)

print("\nLeaked feature removed: clk_mar")
print("Honest ROC-AUC kept:", honest_auc)

Final features:
- impressions_feb
- clicks_feb
- ctr_feb
- avg_position_feb
- gsc_days_available_feb

Leaked feature removed: clk_mar
Honest ROC-AUC kept: 0.8222124928456469


### Why these five features are available at the decision moment

- impressions_feb — available because February search impressions were already observed by February 28.
- clicks_feb — available because February search clicks were already observed by February 28.
- ctr_feb — available because it is calculated only from February impressions and clicks.
- avg_position_feb — available because it is calculated from February search-position observations.
- gsc_days_available_feb — available because GSC availability during February was already known by February 28.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitation

The warehouse data is observational, so it can show patterns in search performance but cannot prove that a content change caused a performance change. Some rows may also have missing Google Search Console availability, so availability must be checked before interpreting search-performance values. The February feature window and March label window must remain separate to avoid leakage.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.